# Flanker Analysis in Nilearn — Part 3: Third-Level (Group) Analysis

The **third-level** (group) analysis asks whether an effect is consistent across **subjects**, so that we can say something about the **population** they were drawn from. Each subject contributes one number per voxel (their subject-level effect from Part 2), and we test whether the mean of those numbers differs from zero.

**Before you start:** you need to have run **Parts 1 and 2**.

**What you will do:**
1. Run a one-sample group test for each of the four contrasts, with and without spatial smoothing (5 mm).
2. Look at the resulting z-maps (no statistical thresholding yet).

**Important:** this dataset has only **3 subjects**, which is far too few for a real group analysis. You will use it to learn the mechanics and to think about why sample size matters.

**What to submit:** answers to the 3 reflection questions at the end, in a separate Word or text file, uploaded to Canvas.

## Step 0 — Tell the notebook where your data are
Use the **same three paths** as in Parts 1 and 2.

In [ ]:
# ---- EDIT THESE THREE PATHS, THEN RUN THE REST OF THE NOTEBOOK UNCHANGED ----
BIDS_DIR     = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/rawdata"                       # contains sub-01/func/*_events.tsv
FMRIPREP_DIR = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/derivatives/fmriprep_outputs/"  # contains sub-01/func/*_desc-preproc_bold.nii.gz
OUTPUT_DIR   = "/Users/mexico/Documents/Teaching/fundamentals_human_brain_imaging/datasets/flanker/derivatives/nilearn_outputs"   # where YOUR results will be saved

## Step 1 — Imports and settings (basically the same as parts 1 and 2)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import nilearn

BIDS_DIR, FMRIPREP_DIR, OUTPUT_DIR = Path(BIDS_DIR), Path(FMRIPREP_DIR), Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Fixed settings for this assignment (you do not need to change these) ----
SUBS = ["sub-01", "sub-02", "sub-03"]
RUNS = ["run-1", "run-2"]
SPACE = "space-MNI152NLin2009cAsym_res-2"

# Contrast name -> {design-matrix column: weight}.  Andy's three contrasts, plus all>baseline.
# (all>baseline uses 0.5/0.5 so the result is the AVERAGE of the two conditions.)
CONTRASTS = {
    "incongruent":           {"incongruent": 1.0},
    "congruent":             {"congruent": 1.0},
    "incongruent-congruent": {"incongruent": 1.0, "congruent": -1.0},
    "all-baseline":          {"incongruent": 0.5, "congruent": 0.5},
}

# ---- Where files live (layout inside each folder follows BIDS / fMRIPrep) ----
def events_path(sub, run):
    return BIDS_DIR / sub / "func" / f"{sub}_task-flanker_{run}_events.tsv"

def fmriprep_path(sub, run, suffix):
    return FMRIPREP_DIR / sub / "func" / f"{sub}_task-flanker_{run}_{suffix}"

def bold_path(sub, run):  return fmriprep_path(sub, run, f"{SPACE}_desc-preproc_bold.nii.gz")
def mask_path(sub, run):  return fmriprep_path(sub, run, f"{SPACE}_desc-brain_mask.nii.gz")
def confounds_path(sub, run): return fmriprep_path(sub, run, "desc-confounds_timeseries.tsv")

# ---- Where this assignment writes its outputs ----
STAT_KEYS = {"effect": "effect_size", "variance": "effect_variance", "z": "z_score"}

def level1_path(sub, run, contrast, stat):
    return OUTPUT_DIR / "level1" / sub / f"{sub}_{run}_contrast-{contrast}_{stat}.nii.gz"

def level2_path(sub, contrast, stat):
    return OUTPUT_DIR / "level2" / sub / f"{sub}_contrast-{contrast}_{stat}.nii.gz"

def level3_path(contrast, smoothing, stat):
    return OUTPUT_DIR / "level3" / f"group_contrast-{contrast}_smooth-{smoothing}_{stat}.nii.gz"


from nilearn.glm.second_level import SecondLevelModel
from nilearn.masking import intersect_masks, apply_mask
from nilearn.plotting import plot_stat_map, plot_glass_brain

SMOOTHING = {"none": None, "5mm": 5.0}     # label -> smoothing FWHM in mm

In [ ]:
missing = [level2_path(s, c, "effect") for s in SUBS for c in CONTRASTS if not level2_path(s, c, "effect").exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} Part 2 outputs are missing. Run Parts 1 and 2 first. Example: {missing[0]}")
print("Part 2 outputs found.")

### How does this differ from the FSL analysis? (conceptual)

| Step | FSL / FEAT (Andy's Brain Book) | Nilearn (this notebook) |
|---|---|---|
| Inputs | Subject-level `cope` images (and their variances) | Subject-level effect-size images |
| Model | **FLAME 1** mixed effects: weights each subject by their *within-subject* uncertainty and estimates the *between-subject* variance | **One-sample t-test (OLS)** on the subject effect maps: every subject counts equally, within-subject variance is not used |
| Multiple-comparison correction | Cluster correction based on Gaussian random field theory | **None yet** — maps are shown unthresholded |
| Smoothing | No smoothing | Applied *here*, at the group stage, as an option (none vs. 5 mm) |

## Step 2 — The group model

The group model is a **one-sample t-test**: the design matrix is a single column of ones (an *intercept*), one row per subject. The estimated intercept is the group mean effect, and the t-test asks whether it differs from zero. The test statistic is converted to a z-score so it is comparable across analyses.

For each contrast we run the test twice: once on the maps as they are, and once after smoothing them with a 5 mm FWHM Gaussian kernel. The group brain mask is the set of voxels inside the brain mask for **all** subjects and runs.

In [ ]:
group_mask = intersect_masks([str(mask_path(s, r)) for s in SUBS for r in RUNS], threshold=1)

design = pd.DataFrame({"intercept": np.ones(len(SUBS))}, index=SUBS)
print("Group design matrix (one row per subject):")
display(design)
print(f"Degrees of freedom for the group t-test: {len(SUBS)} - 1 = {len(SUBS) - 1}")

GROUP_Z = {}
level3_path("incongruent", "none", "z").parent.mkdir(parents=True, exist_ok=True)

for cname in CONTRASTS:
    effect_imgs = [nib.load(str(level2_path(sub, cname, "effect"))) for sub in SUBS]
    for label, fwhm in SMOOTHING.items():
        slm = SecondLevelModel(mask_img=group_mask, smoothing_fwhm=fwhm)
        slm.fit(effect_imgs, design_matrix=design)
        z_img = slm.compute_contrast("intercept", second_level_stat_type="t", output_type="z_score")
        z_img.to_filename(str(level3_path(cname, label, "z")))
        GROUP_Z[(cname, label)] = z_img
    print(f"{cname}: done")

## Step 3 — Look at the group maps

For each contrast, the left panel is the group z-map **without smoothing**, the right panel is **with 5 mm smoothing**. There is no statistical threshold, so every voxel in the brain is colored by its z-value. Notice the different color range needed for different contrasts.

Things to look for: how does smoothing change the appearance of the map? Which contrast has the strongest, most spatially coherent signal?

In [ ]:
for cname in CONTRASTS:
    fig, axes = plt.subplots(1, len(SMOOTHING), figsize=(14, 3.4))
    for ax, label in zip(axes, SMOOTHING):
        plot_stat_map(GROUP_Z[(cname, label)], axes=ax, display_mode="z", cut_coords=[20, 35, 50],
                      title=f"{cname} - smoothing: {label}")
    plt.show()

## Step 4 — What do the z-values look like overall?

A histogram of every z-value in the brain for the **incongruent − congruent** contrast. If there were no effects at all, we would expect values spread out like a standard normal distribution (mean 0, standard deviation 1). How does the observed distribution compare, and how does smoothing change it?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for label in SMOOTHING:
    vals = apply_mask(GROUP_Z[("incongruent-congruent", label)], group_mask)
    ax.hist(vals, bins=100, alpha=0.5, density=True, label=f"smoothing: {label}  (SD = {vals.std():.2f})")
ax.set(xlabel="group z-value", ylabel="density", title="Distribution of group z-values: incongruent - congruent")
ax.legend()
plt.tight_layout()

## A note on statistical inference with more participants

You have just estimated *unthresholded* group maps, and with only 3 subjects the test has just **2 degrees of freedom**: the results are very noisy and the normality assumption behind the t-to-z conversion is shaky. With a real sample (for example, all 26 subjects in this dataset), you would typically make inferences using a **permutation test** — in Nilearn, `nilearn.glm.second_level.non_parametric_inference`. For a one-sample test, it randomly flips the sign of each subject's map many times to build a null distribution without assuming normality, and it can control the family-wise error rate across the whole brain. With only 3 subjects there are just 2³ = 8 possible sign-flips, so the smallest possible p-value is 1/8 = 0.125 and no voxel could ever be significant — which is why we are not running it here. We will come back to thresholding and multiple-comparison correction later.

## Reflection questions

Write your answers in a **separate Word document or text file** (not in this notebook) and upload it to Canvas. A few sentences for each is enough.

1. **Sample size.** The group test here has 2 degrees of freedom. What does this mean for how much you should trust a z-map from this analysis, and what would you expect to change with 26 subjects? Explain, in your own words, how a permutation (sign-flipping) test works and why it is a useful alternative to the t-test when you cannot trust distributional assumptions.
2. **Smoothing at the group level.** Compare the smoothed and unsmoothed maps and the histogram. What did smoothing do to the maps and to the distribution of z-values? Give two reasons why researchers commonly smooth fMRI data before group analysis.
3. **Reflect on the process of Nilearn versus FSL** for the level1, level2, and level3 models. What are the pros and cons of each? Which one would you be more likely to use in the future (for example, for your project, if you are doing task-based analysis)?